In [1]:
import hashlib
import urllib.request

import duckdb
import pandas as pd
import geopandas as gpd
from pathlib import Path

## Source

8 quarters of Ookla mobile tiles (2024 Q3 → 2026 Q2), fetched directly from the public
`ookla-open-data` S3 bucket (no AWS account needed) and clipped to the real UAE boundary polygon
(never a lon/lat bounding box — see [`00_ookla_dataset_overview.ipynb`](00_ookla_dataset_overview.ipynb)
for why). Each row in the final output is tagged with the quarter it came from, since individual
tiles churn heavily across quarters and can't be safely joined on `quadkey` alone.

In [2]:
BOUNDARY_URL = (
    "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/"
    "releaseData/gbOpen/ARE/ADM0/geoBoundaries-ARE-ADM0.geojson"
)
boundary_file = Path("../data/raw/boundary/uae_boundary.geojson")
boundary_file.parent.mkdir(parents=True, exist_ok=True)

if boundary_file.exists():
    print("Already downloaded:", boundary_file)
else:
    print("Downloading UAE boundary (geoBoundaries.org, ADM0):", BOUNDARY_URL)
    urllib.request.urlretrieve(BOUNDARY_URL, boundary_file)
    print("Saved to:", boundary_file)

uae_boundary = gpd.read_file(boundary_file).to_crs("EPSG:4326")

RAW_DIR = Path("../data/raw/ookla")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# (year, quarter, quarter start date) for the 8 quarters this project uses:
# latest published quarter (2026 Q2) plus the previous 7.
QUARTERS = [
    (2024, 3, "2024-07-01"),
    (2024, 4, "2024-10-01"),
    (2025, 1, "2025-01-01"),
    (2025, 2, "2025-04-01"),
    (2025, 3, "2025-07-01"),
    (2025, 4, "2025-10-01"),
    (2026, 1, "2026-01-01"),
    (2026, 2, "2026-04-01"),
]

def s3_url(year, quarter, date_str):
    return (
        "https://ookla-open-data.s3.amazonaws.com/parquet/performance/"
        f"type=mobile/year={year}/quarter={quarter}/"
        f"{date_str}_performance_mobile_tiles.parquet"
    )

def quarter_label(year, quarter):
    return f"{year}Q{quarter}"

Saved to: ..\data\raw\boundary\uae_boundary.geojson


In [3]:
# Download each quarter (skip if already present). ~150-250 MB total, so this can
# take a few minutes on a cold cache.

ookla_files = []

for year, quarter, date_str in QUARTERS:
    local_path = RAW_DIR / f"{date_str}_performance_mobile_tiles.parquet"

    if local_path.exists():
        print("Already downloaded:", local_path.name)
    else:
        url = s3_url(year, quarter, date_str)
        print("Downloading:", url)
        urllib.request.urlretrieve(url, local_path)
        print("Saved to:", local_path)

    ookla_files.append(local_path)

for f in ookla_files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{f.name}: {size_mb:.1f} MB")

Already downloaded: 2024-07-01_performance_mobile_tiles.parquet
Already downloaded: 2024-10-01_performance_mobile_tiles.parquet
Already downloaded: 2025-01-01_performance_mobile_tiles.parquet
Already downloaded: 2025-04-01_performance_mobile_tiles.parquet
Already downloaded: 2025-07-01_performance_mobile_tiles.parquet
Already downloaded: 2025-10-01_performance_mobile_tiles.parquet
Already downloaded: 2026-01-01_performance_mobile_tiles.parquet
Already downloaded: 2026-04-01_performance_mobile_tiles.parquet
2024-07-01_performance_mobile_tiles.parquet: 197.3 MB
2024-10-01_performance_mobile_tiles.parquet: 186.1 MB
2025-01-01_performance_mobile_tiles.parquet: 177.9 MB
2025-04-01_performance_mobile_tiles.parquet: 185.1 MB
2025-07-01_performance_mobile_tiles.parquet: 180.0 MB
2025-10-01_performance_mobile_tiles.parquet: 173.9 MB
2026-01-01_performance_mobile_tiles.parquet: 167.2 MB
2026-04-01_performance_mobile_tiles.parquet: 176.0 MB


In [4]:
def extract_uae_tiles(file, quarter):

    # Step 1: rough geographic pre-filter
    candidate = duckdb.sql(f"""
        SELECT *
        FROM read_parquet('{file}')
        WHERE tile_x BETWEEN 51 AND 57
          AND tile_y BETWEEN 22 AND 27
    """).df()

    # Step 2: convert centroids to geographic points
    points = gpd.GeoDataFrame(
        candidate,
        geometry=gpd.points_from_xy(
            candidate["tile_x"],
            candidate["tile_y"]
        ),
        crs="EPSG:4326"
    )

    # Step 3: exact UAE polygon filter
    uae = gpd.sjoin(
        points,
        uae_boundary[["geometry"]],
        predicate="within",
        how="inner"
    )

    # remove spatial-join helper column
    if "index_right" in uae.columns:
        uae = uae.drop(columns=["index_right"])

    # tag with the quarter this snapshot came from -- tiles churn heavily
    # across quarters (see 00_ookla_dataset_overview.ipynb), so downstream
    # quarter-over-quarter comparisons need this to distinguish otherwise
    # identical quadkeys.
    uae["quarter"] = quarter

    # the raw "tile" column is the tile's actual polygon boundary (WKT), not
    # just its centroid. Parse it into a real geometry column instead of
    # leaving it as an unusable string -- it's more precise than the
    # synthetic centroid point for area/edge calculations later.
    uae["tile_geometry"] = gpd.GeoSeries.from_wkt(uae["tile"], crs="EPSG:4326")
    uae = uae.drop(columns=["tile"])

    return uae

In [5]:
# Process all 8

all_quarters = []

for (year, quarter, date_str), file in zip(QUARTERS, ookla_files):

    label = quarter_label(year, quarter)
    print("Processing:", file.name, f"({label})")

    uae = extract_uae_tiles(file, label)

    print("UAE tiles:", len(uae))

    all_quarters.append(uae)

Processing: 2024-07-01_performance_mobile_tiles.parquet (2024Q3)
UAE tiles: 5818
Processing: 2024-10-01_performance_mobile_tiles.parquet (2024Q4)


UAE tiles: 6023
Processing: 2025-01-01_performance_mobile_tiles.parquet (2025Q1)
UAE tiles: 5965
Processing: 2025-04-01_performance_mobile_tiles.parquet (2025Q2)


UAE tiles: 5982
Processing: 2025-07-01_performance_mobile_tiles.parquet (2025Q3)
UAE tiles: 6187
Processing: 2025-10-01_performance_mobile_tiles.parquet (2025Q4)


UAE tiles: 6844
Processing: 2026-01-01_performance_mobile_tiles.parquet (2026Q1)
UAE tiles: 7162
Processing: 2026-04-01_performance_mobile_tiles.parquet (2026Q2)


UAE tiles: 6879


In [6]:
# Combine all 8 
ookla_uae = pd.concat(
    all_quarters,
    ignore_index=True
)

In [7]:
print("Shape:", ookla_uae.shape)
print()
print("Rows per quarter:")
print(ookla_uae["quarter"].value_counts().sort_index())
print()

# Sanity check: the fix for the quarter-labelling gap. Quadkeys still repeat
# across quarters (tiles that persist), but (quadkey, quarter) should now be
# unique -- unlike before, when duplicate quadkeys across quarters were
# silently indistinguishable.
dup_quadkey_only = ookla_uae.duplicated(subset=["quadkey"]).sum()
dup_quadkey_and_quarter = ookla_uae.duplicated(subset=["quadkey", "quarter"]).sum()
print(f"Rows sharing a quadkey with another row (expected, across quarters): {dup_quadkey_only}")
print(f"Rows sharing (quadkey, quarter) with another row (should be 0): {dup_quadkey_and_quarter}")

Shape: (50860, 13)

Rows per quarter:
quarter
2024Q3    5818
2024Q4    6023
2025Q1    5965
2025Q2    5982
2025Q3    6187
2025Q4    6844
2026Q1    7162
2026Q2    6879
Name: count, dtype: int64

Rows sharing a quadkey with another row (expected, across quarters): 35659
Rows sharing (quadkey, quarter) with another row (should be 0): 0


In [8]:
Path("../data/processed").mkdir(exist_ok=True)

processed_path = Path("../data/processed/ookla_tiles_uae.parquet")

ookla_uae.to_parquet(processed_path, index=False)

sha256 = hashlib.sha256(processed_path.read_bytes()).hexdigest()
size_mb = processed_path.stat().st_size / (1024 ** 2)

print(f"Saved: {processed_path} ({size_mb:.1f} MB)")
print(f"SHA256: {sha256}")
print()
print("NOTE: this file has two GeoParquet geometry columns (`geometry` = tile")
print("centroid point, `tile_geometry` = real tile polygon boundary). Read it")
print("with geopandas.read_parquet(...), not pandas.read_parquet(...) -- plain")
print("pandas returns raw WKB bytes for both geometry columns instead of usable")
print("geometries, with no error to warn you.")

Saved: ..\data\processed\ookla_tiles_uae.parquet (2.3 MB)
SHA256: 32b3c2259685027db6c2a8ff86cfba463c8c462c2970f4ce49a19617cfcc4182

NOTE: this file has two GeoParquet geometry columns (`geometry` = tile
centroid point, `tile_geometry` = real tile polygon boundary). Read it
with geopandas.read_parquet(...), not pandas.read_parquet(...) -- plain
pandas returns raw WKB bytes for both geometry columns instead of usable
geometries, with no error to warn you.
